## Librerias


In [44]:
import numpy as np
import trimesh
from trimesh.transformations import rotation_matrix
from utils.noise import add_sensor_noise, add_background_noise, add_poisson_noise
import os
import plotly.graph_objects as go
from utils.check_overlaps import check_overlaps
from utils.create_wall import create_wall_mesh
from utils.get_files import get_obj_files
from utils.sparse_wall import create_sparse_wall
from utils.densify_mesh import densify_mesh_if_needed


## ⚠️⚠️ Parametros no modificables ⚠️⚠️


In [18]:
c = 299792458

object_folder = 'objects'

ymin = 0
zmin = 0

## ✅✅ Parámetros de entrada (modificables) ✅✅


In [19]:
xmin = -1.5
xmax = 1.5
ymax = 3
zmax = 3

camera_FOV = 0.25
cam_pixel_dim = 64
bin_size = 3.9e-10
laser_intensity = 1000
hide_walls = False
SNR_dB = 30
SBR = 5
poisson_scale_factor = 1000
add_noise = False
MESH_MIN_TRIANGLES = 5000

# polar: x = rho*cos(phi), y = rho*sin(phi); object on ground (z=0)
object_positions = [
    {
        'obj_file': 'facet.obj',
        'rho': 0.5,
        'phi_deg': 120,
        'w': 0.5,
        'h': 1.0,
        'yaw': 0,
        'pitch': 1.57,
        'roll': 0,
    }
]


# Función principal de simulación parametrizada en coordenadas polares


In [20]:
def polar_to_scene_xy(rho, phi=None, phi_deg=None):
    """Return (x, y, phi_rad) with x=rho*cos(phi), y=rho*sin(phi)."""
    rho = float(rho)
    if rho <= 0:
        raise ValueError(f"rho must be > 0, got {rho}")
    if phi is None and phi_deg is None:
        raise ValueError("Provide phi or phi_deg")
    if phi is not None and phi_deg is not None:
        raise ValueError("Provide only one of phi / phi_deg")
    if phi_deg is not None:
        phi = np.deg2rad(float(phi_deg))
    else:
        phi = float(phi)
    return rho * np.cos(phi), rho * np.sin(phi), phi


In [45]:
def simulation(
    xmin, xmax, ymax, zmax,
    camera_FOV, cam_pixel_dim, bin_size, laser_intensity,
    object_positions, hide_walls,
    SNR_dB, SBR, poisson_scale_factor, add_noise,
    uploaded_objs=None,
    subpixel_dim=1,
):
    objects = []
    scene_objects = []

    camera_FOV_center = [0, -camera_FOV / 2, 0]
    FOV_radius = camera_FOV / cam_pixel_dim

    laser_pos = np.array([0, 0, 0], dtype=float)
    laser_normal = np.array([0, 0, 1], dtype=float)

    wall_discr = c / 2 * bin_size / 4

    params = {
        "cam_pixel_dim": cam_pixel_dim,
        "camera_FOV": camera_FOV,
        "camera_FOV_center": camera_FOV_center,
        "FOV_radius": FOV_radius,
        "laser_intensity": laser_intensity,
        "bin_size": bin_size,
        "c": c,
        "laser_pos": laser_pos,
        "laser_normal": laser_normal,
        "wall_discr": wall_discr,
        "subpixel_dim": subpixel_dim,
    }

    furthest_scene_point = np.array([xmax, ymax, zmax], dtype=float)
    furthest_spad_point = np.array([-camera_FOV / 2, -camera_FOV, 0], dtype=float)

    d1 = np.linalg.norm(furthest_scene_point - laser_pos)
    d2 = np.linalg.norm(furthest_spad_point - furthest_scene_point)
    max_dist_travel = d1 + d2

    num_time_bins = int(np.ceil(1.2 * max_dist_travel / (c * bin_size)))
    params["num_time_bins"] = num_time_bins

    if not hide_walls:
        back_wall = create_wall_mesh(
            np.array([xmin, ymax, 0]),
            np.array([xmax - xmin, 0, 0]),
            np.array([0, 0, zmax]),
        )
        right_wall = create_wall_mesh(
            np.array([xmax, ymax, 0]),
            np.array([0, -ymax, 0]),
            np.array([0, 0, zmax]),
        )
        left_wall = create_wall_mesh(
            np.array([xmin, ymax, 0]),
            np.array([0, -ymax, 0]),
            np.array([0, 0, zmax]),
        )
        ceiling = create_wall_mesh(
            np.array([xmin, ymax, zmax]),
            np.array([xmax - xmin, 0, 0]),
            np.array([0, -ymax, 0]),
        )

        back_wall = densify_mesh_if_needed(back_wall, min_triangles=MESH_MIN_TRIANGLES)
        right_wall = densify_mesh_if_needed(right_wall, min_triangles=MESH_MIN_TRIANGLES)
        left_wall = densify_mesh_if_needed(left_wall, min_triangles=MESH_MIN_TRIANGLES)
        ceiling = densify_mesh_if_needed(ceiling, min_triangles=MESH_MIN_TRIANGLES)

        objects.extend([back_wall, right_wall, left_wall, ceiling])

    front_wall_spheres = create_sparse_wall(
        origin=np.array([xmin, ymin, zmin]),
        width_vec=np.array([0 - xmin, 0, 0]),
        height_vec=np.array([0, 0, zmax - zmin]),
        color=[200, 200, 200, 255],
        sphere_radius=0.015,
        spacing=0.3,
    )

    for obj_data in object_positions:
        obj_file = obj_data["obj_file"]
        rho = float(obj_data["rho"])

        if "phi_deg" in obj_data:
            xcoord, ycoord, phi_rad = polar_to_scene_xy(rho=rho, phi_deg=obj_data["phi_deg"])
        elif "phi" in obj_data:
            xcoord, ycoord, phi_rad = polar_to_scene_xy(rho=rho, phi=obj_data["phi"])
        else:
            raise KeyError("Each object must have either 'phi' or 'phi_deg'.")

        zcoord = 0.0
        w = float(obj_data["w"])
        h = float(obj_data.get("h", 1.0))
        pitch = float(obj_data.get("pitch", 1.57))
        roll = float(obj_data.get("roll", 0.0))

        v1 = np.array([xcoord, ycoord, zcoord], dtype=float)

        print(
            f"Objeto {obj_file}: rho={rho:.4f}, "
            f"phi={phi_rad:.4f} rad ({np.rad2deg(phi_rad):.2f} deg) -> "
            f"x={xcoord:.4f}, y={ycoord:.4f}, z={zcoord:.4f}, w={w:.4f}, h={h:.4f}"
        )

        if uploaded_objs and obj_file.startswith("uploaded_"):
            uploaded_file = uploaded_objs[obj_file]
            uploaded_file.seek(0)
            obj = trimesh.load(uploaded_file, file_type="obj", force="mesh")
            if isinstance(obj, trimesh.Scene):
                obj = obj.dump(concatenate=True)
        else:
            obj_path = os.path.join(object_folder, obj_file)
            obj = trimesh.load(obj_path, force="mesh")

        obj = densify_mesh_if_needed(obj, min_triangles=MESH_MIN_TRIANGLES)

        obj_extents = obj.extents
        if obj_extents[0] <= 0:
            raise ValueError(f"Invalid object X extent: {obj_extents}")
        if obj_extents[1] <= 0:
            raise ValueError(
                f"Invalid object Y extent for height scaling: {obj_extents}"
            )

        # Anisotropic scale before pitch: X -> width w, Y -> height h
        # (pitch ~ pi/2 maps mesh Y into world Z).
        scale_x = w / obj_extents[0]
        scale_y = h / obj_extents[1]
        obj.apply_scale([scale_x, scale_y, 1.0])

        obj.apply_transform(rotation_matrix(pitch, [1, 0, 0]))
        obj.apply_transform(rotation_matrix(roll, [0, 1, 0]))

        # Corrected orientation for your facet.obj:
        # makes mean normal align with [-cos(phi), -sin(phi), 0]
        theta = phi_rad + 3 * np.pi / 2
        obj.apply_transform(rotation_matrix(theta, [0, 0, 1]))

        mean_normal = obj.face_normals.mean(axis=0)
        if np.linalg.norm(mean_normal) > 0:
            mean_normal = mean_normal / np.linalg.norm(mean_normal)
            expected_normal = np.array([-np.cos(phi_rad), -np.sin(phi_rad), 0.0])
            print("mean facet normal:", mean_normal)
            print("expected normal:", expected_normal)
            print("alignment dot:", np.dot(mean_normal, expected_normal))
            print()

        z_min = obj.vertices[:, 2].min()
        obj.apply_translation([0, 0, -z_min])
        obj.apply_translation(v1)

        z_span = float(obj.vertices[:, 2].max() - obj.vertices[:, 2].min())
        xy = obj.vertices[:, :2]
        # approximate horizontal span after placement
        print(f"  placed facet Z span ≈ {z_span:.4f} m (target h={h:.4f})")

        scene_objects.append(obj)

    objects.extend(scene_objects)

    # ------------------------------------------------------------
    # Pixel grid with subpixel sampling
    # ------------------------------------------------------------
    pixel_pitch = camera_FOV / cam_pixel_dim

    pixel_x_centers = np.linspace(
        camera_FOV_center[0] - camera_FOV / 2 + pixel_pitch / 2,
        camera_FOV_center[0] + camera_FOV / 2 - pixel_pitch / 2,
        cam_pixel_dim,
    )

    pixel_y_centers = np.linspace(
        camera_FOV_center[1] - camera_FOV / 2 + pixel_pitch / 2,
        camera_FOV_center[1] + camera_FOV / 2 - pixel_pitch / 2,
        cam_pixel_dim,
    )

    if subpixel_dim <= 1:
        sub_offsets = np.array([0.0])
    else:
        sub_offsets = (np.arange(subpixel_dim) + 0.5) / subpixel_dim - 0.5

    sub_offsets = sub_offsets * pixel_pitch

    cam_pos_list = []
    cam_pos_ind_list = []

    for iy, yc in enumerate(pixel_y_centers):
        for ix, xc in enumerate(pixel_x_centers):
            for oy in sub_offsets:
                for ox in sub_offsets:
                    cam_pos_list.append([xc + ox, yc + oy, 0.0])
                    cam_pos_ind_list.append([ix, iy])

    cam_pos = np.asarray(cam_pos_list, dtype=float)
    cam_pos_ind = np.asarray(cam_pos_ind_list, dtype=int)

    num_subsamples = subpixel_dim * subpixel_dim

    y_meas_vec = np.zeros(
        cam_pixel_dim * cam_pixel_dim * num_time_bins,
        dtype=float,
    )

    if len(objects) > 0:
        combined_mesh = trimesh.util.concatenate(objects)
        triangles = combined_mesh.triangles
        triangle_normals = combined_mesh.face_normals

        fourpi = 4 * np.pi * np.pi
        floor_normal = np.array([0, 0, 1], dtype=float)

        for idx in range(len(triangles)):
            triangle = triangles[idx]
            normal = triangle_normals[idx]

            area = 0.5 * np.linalg.norm(
                np.cross(
                    triangle[1] - triangle[0],
                    triangle[2] - triangle[0],
                )
            )

            if area <= 0:
                continue

            scene_center = triangle.mean(axis=0)

            denom = scene_center[0] - cam_pos[:, 0]
            valid_denom = np.abs(denom) > 1e-12

            m = np.full(cam_pos.shape[0], np.nan)
            m[valid_denom] = (
                scene_center[1] - cam_pos[valid_denom, 1]
            ) / denom[valid_denom]

            b = scene_center[1] - m * scene_center[0]

            xint = np.full(cam_pos.shape[0], np.nan)
            valid_m = np.abs(m) > 1e-12
            xint[valid_m] = -b[valid_m] / m[valid_m]

            noc = np.isfinite(xint) & (xint > 0)

            if np.sum(noc) == 0:
                continue

            lps = laser_pos - scene_center
            fovsp = cam_pos[noc, :] - scene_center

            d1s = np.sum(lps**2)
            d2s = np.sum(fovsp**2, axis=1)

            if d1s <= 0:
                continue

            d1 = np.sqrt(d1s)
            d2 = np.sqrt(d2s)

            valid_d2 = d2 > 0
            if np.sum(valid_d2) == 0:
                continue

            fovsp = fovsp[valid_d2]
            d2 = d2[valid_d2]
            d2s = d2s[valid_d2]
            noc_indices = np.where(noc)[0][valid_d2]

            distance = d1 + d2
            arrival_bin = np.ceil(distance / (c * bin_size)).astype(int)

            valid_bins = (arrival_bin >= 1) & (arrival_bin <= num_time_bins)
            if np.sum(valid_bins) == 0:
                continue

            arrival_bin = arrival_bin[valid_bins]
            fovsp = fovsp[valid_bins]
            d2 = d2[valid_bins]
            d2s = d2s[valid_bins]
            noc_indices = noc_indices[valid_bins]

            dot1 = max(0.0, np.dot(normal, lps / d1))

            dot2 = np.maximum(
                0.0,
                np.sum(normal * fovsp / d2[:, None], axis=1),
            )

            dot3 = max(
                0.0,
                np.dot(laser_normal, -lps / d1),
            )

            dot4 = np.maximum(
                0.0,
                np.sum(floor_normal * -fovsp / d2[:, None], axis=1),
            )

            intensity = (
                laser_intensity
                * area
                * dot1
                * dot2
                * dot3
                * dot4
                / (fourpi * d1s * d2s)
            )

            # Average subpixel samples back into the original pixel
            intensity = intensity / num_subsamples

            px = cam_pos_ind[noc_indices, 0]
            py = cam_pos_ind[noc_indices, 1]
            tb = arrival_bin - 1

            coord = tb * cam_pixel_dim**2 + px * cam_pixel_dim + py
            np.add.at(y_meas_vec, coord, intensity)

    y_meas_vec = y_meas_vec.reshape(
        (cam_pixel_dim, cam_pixel_dim, num_time_bins),
        order="F",
    )

    if add_noise:
        y_with_background = add_background_noise(y_meas_vec, sbr=SBR)
        y_with_shot_noise = add_poisson_noise(
            y_with_background,
            scale_factor=poisson_scale_factor,
        )
        y_meas_vec_noisy = add_sensor_noise(y_with_shot_noise, SNR_dB)
    else:
        y_meas_vec_noisy = y_meas_vec

    fig3d = go.Figure()

    for mesh in objects:
        vertices = mesh.vertices
        faces = mesh.faces
        x, y, z = vertices.T
        i, j, k_faces = faces.T

        fig3d.add_trace(
            go.Mesh3d(
                x=x,
                y=y,
                z=z,
                i=i,
                j=j,
                k=k_faces,
                color="blue",
                opacity=1.0,
                name="Walls & Ceiling",
            )
        )

    for idx, obj in enumerate(scene_objects):
        fig3d.add_trace(
            go.Mesh3d(
                x=obj.vertices[:, 0],
                y=obj.vertices[:, 1],
                z=obj.vertices[:, 2],
                i=obj.faces[:, 0],
                j=obj.faces[:, 1],
                k=obj.faces[:, 2],
                color="orange",
                opacity=1.0,
                name=f"Object {idx + 1}",
            )
        )

    # Plot only pixel centers, not all subpixels
    Xc, Yc = np.meshgrid(pixel_x_centers, pixel_y_centers, indexing="xy")
    cam_centers = np.vstack(
        [
            Xc.ravel(),
            Yc.ravel(),
            np.zeros(cam_pixel_dim**2),
        ]
    ).T

    fig3d.add_trace(
        go.Scatter3d(
            x=cam_centers[:, 0],
            y=cam_centers[:, 1],
            z=cam_centers[:, 2],
            mode="markers",
            marker=dict(size=3, color="green", opacity=0.8),
            name="Camera Pixels",
            showlegend=False,
        )
    )

    laser_sphere = trimesh.creation.icosphere(radius=0.04)
    laser_sphere.apply_translation(laser_pos)

    fig3d.add_trace(
        go.Mesh3d(
            x=laser_sphere.vertices[:, 0],
            y=laser_sphere.vertices[:, 1],
            z=laser_sphere.vertices[:, 2],
            i=laser_sphere.faces[:, 0],
            j=laser_sphere.faces[:, 1],
            k=laser_sphere.faces[:, 2],
            color="red",
            opacity=1.0,
            name="Laser",
            showscale=False,
        )
    )

    front_wall_x = []
    front_wall_y = []
    front_wall_z = []

    for sphere in front_wall_spheres:
        front_wall_x.extend(sphere.vertices[:, 0])
        front_wall_y.extend(sphere.vertices[:, 1])
        front_wall_z.extend(sphere.vertices[:, 2])

    fig3d.add_trace(
        go.Scatter3d(
            x=front_wall_x,
            y=front_wall_y,
            z=front_wall_z,
            mode="markers",
            marker=dict(size=3, color="dodgerblue", opacity=0.6),
            name="Occluding Wall",
            showlegend=False,
        )
    )

    fig3d.update_layout(
        scene=dict(
            xaxis_title="X",
            yaxis_title="Y",
            zaxis_title="Z",
            aspectmode="data",
            camera=dict(
                eye=dict(x=-1.5, y=-1.5, z=1),
                center=dict(x=0, y=0, z=0),
                up=dict(x=0, y=0, z=1),
            ),
        ),
        title="3D Scene Visualization",
        width=800,
        height=800,
    )

    return fig3d, y_meas_vec_noisy, params


# 🐇 Ejecutar la simulación 🐇


In [22]:
# Ejecutar la simulación
fig3d, y_meas_vec_noisy, params = simulation(
    xmin, xmax, ymax, zmax,
    camera_FOV, cam_pixel_dim, bin_size, laser_intensity,
    object_positions, hide_walls,
    SNR_dB, SBR, poisson_scale_factor,
    add_noise
)


Objeto facet.obj: rho=0.5000, phi=2.0944 rad (120.00 deg) -> x=-0.2500, y=0.4330, z=0.0000
mean facet normal: [ 4.99999841e-01 -8.66025129e-01  7.96326711e-04]
expected normal: [ 0.5       -0.8660254  0.       ]
alignment dot: 0.9999996829318346



## Visualización de la escena 3D

In [23]:
fig3d.show()

# Visualización de los resultados de la simulación

In [24]:
# Suma la intensidad integrada a lo largo del eje temporal
y_sum = np.sum(y_meas_vec_noisy, axis=2)
# Aplica un corrimiento para ajustar la posición de la imagen

# Define el origen en X (centro) y en Y (última fila)
origin_x = y_sum.shape[1] // 2
origin_y = y_sum.shape[0] - 1

# Ajusta las coordenadas para que el origen quede centrado en X y en la parte inferior en Y
adjusted_x = np.arange(y_sum.shape[1]) - origin_x  
adjusted_y = np.arange(y_sum.shape[0]) - origin_y

# Crea la figura del mapa de calor de la intensidad integrada en el tiempo usando Plotly
fig_intensity = go.Figure(data=go.Heatmap(
    x=adjusted_x,
    y=adjusted_y,
    z=y_sum,
    colorscale='Hot',
    colorbar=dict(
        title='Intensidad',
        tickfont=dict(color='black'),
        title_font=dict(color='black')
    )
))

# Configura el layout de la figura (títulos, ejes y escala)
fig_intensity.update_layout(
    title='Intensidad integrada en el tiempo',
    title_font=dict(color='black'),
    xaxis_title='Píxel X',
    xaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black')),
    yaxis_title='Píxel Y',
    yaxis=dict(title_font=dict(color='black'), tickfont=dict(color='black'), scaleanchor="x", scaleratio=1)
)

# Muestra la figura del heatmap (esto abrirá la ventana en el navegador o la renderizará en el notebook)
fig_intensity.show()

# En lugar de usar interactividad con Streamlit, solicitamos al usuario las coordenadas del píxel desde la consola.
print("Ingrese las coordenadas del píxel para inspeccionar (desplazamiento respecto al origen):")
print("(Valor por defecto: x = 0, y = 0, lo que corresponde al centro)")

try:
    # Se solicitan los desplazamientos en X e Y (respecto al origen ajustado)
    user_input_x = input("Coordenada X (desplazamiento): ")
    user_input_y = input("Coordenada Y (desplazamiento): ")
    # Si el usuario ingresa un valor, se convierte a entero; si no, se utiliza el valor por defecto 0
    selected_x_offset = int(user_input_x) if user_input_x.strip() != "" else 0
    selected_y_offset = int(user_input_y) if user_input_y.strip() != "" else 0
except Exception as e:
    print("Entrada inválida, se utilizarán los valores por defecto (0, 0).")
    selected_x_offset = 0
    selected_y_offset = 0

# Calcula las coordenadas del píxel en el arreglo original sumando el desplazamiento al origen
pixel_x = selected_x_offset + origin_x
pixel_y = selected_y_offset + origin_y

# Realiza un corrimiento en el eje 1 de y_meas_vec_noisy (igual que en el heatmap)
y_meas_vec_shifted = np.roll(y_meas_vec_noisy, shift=1, axis=1)
# Obtiene la respuesta temporal para el píxel seleccionado
temporal_response = y_meas_vec_noisy[pixel_y, pixel_x, :]
# Define el eje temporal usando el tamaño del bin almacenado en params
time_axis = np.arange(len(temporal_response)) * params['bin_size']

# Crea la figura de la respuesta temporal con un gráfico de líneas de Plotly
fig_temporal = go.Figure(data=go.Scatter(
    x=time_axis,
    y=temporal_response,
    mode='lines'
))
                                    
fig_temporal.update_layout(
    title=f'Respuesta temporal en el píxel ({selected_x_offset}, {selected_y_offset})',
    xaxis_title='Intervalo de tiempo',
    yaxis_title='Intensidad'
)

# Muestra la gráfica de la respuesta temporal
fig_temporal.show()


Ingrese las coordenadas del píxel para inspeccionar (desplazamiento respecto al origen):
(Valor por defecto: x = 0, y = 0, lo que corresponde al centro)


## CRB Analysis


In [ ]:
from crb_polar_functions import (
    compute_crb_polar,
    compute_crb_grid_polar,
    plot_crb_regions_polar,
)

SIMULATION_KWARGS = dict(
    xmin=xmin,
    xmax=xmax,
    ymax=ymax,
    zmax=zmax,
    camera_FOV=camera_FOV,
    cam_pixel_dim=cam_pixel_dim,
    bin_size=bin_size,
    laser_intensity=laser_intensity,
    object_positions=[],
    hide_walls=True,
    SNR_dB=SNR_dB,
    SBR=SBR,
    poisson_scale_factor=poisson_scale_factor,
    add_noise=False,
    subpixel_dim=4,
)


In [ ]:
facet_width = 0.5
facet_height = 1.0
finite_difference_steps = np.array([0.01, np.deg2rad(0.5), 0.01])  # rho, phi, h
ranges = np.array([0.5, 1.0, 1.5])
angles_deg = np.array([30, 60, 90, 120, 150])

for rho_i in ranges:
    result = compute_crb_polar(
        simulation_fn=simulation,
        simulation_kwargs=SIMULATION_KWARGS,
        rho0=rho_i,
        phi0=np.deg2rad(90),
        width=facet_width,
        height=facet_height,
        finite_difference_steps=finite_difference_steps,
        background_rate=0.01,
        k=3.0,
        force_no_noise=True,
    )
    print(
        f"rho={rho_i}: σρ={result['sigma_rho']:.4g}, "
        f"ρσφ={result['sigma_tangential']:.4g}, "
        f"σh={result['sigma_height']:.4g}"
    )


In [ ]:
# Grid CRB: CRB(rho,phi,h) y CRB(rho,phi)
results_h = compute_crb_grid_polar(
    simulation_fn=simulation,
    simulation_kwargs=SIMULATION_KWARGS,
    ranges=ranges,
    angles_deg=angles_deg,
    width=facet_width,
    height=facet_height,
    estimate_height=True,
    finite_difference_steps=finite_difference_steps,
    background_rate=0.1,
    k=3.0,
    verbose=True,
)
plot_crb_regions_polar(
    results_h,
    use_physical_region=False,
    rlim=2.0,
    title=r"CRB$(\rho,\varphi,h)$ — regiones $3\sigma$ en $(\rho,\varphi)$",
    output_path="plots/crb_standard_regions.png",
)

results_fixed = compute_crb_grid_polar(
    simulation_fn=simulation,
    simulation_kwargs=SIMULATION_KWARGS,
    ranges=ranges,
    angles_deg=angles_deg,
    width=facet_width,
    height=facet_height,
    estimate_height=False,
    finite_difference_steps=finite_difference_steps[:2],
    background_rate=0.1,
    k=3.0,
    verbose=True,
)
plot_crb_regions_polar(
    results_fixed,
    use_physical_region=False,
    rlim=2.0,
    title=r"CRB$(\rho,\varphi)$ — regiones $3\sigma$",
    output_path="plots/crb_standard_regions_fixed_h.png",
)
